 Chapter 4: A/B Testing

Context
A computer company has two email lists: desktop and laptop customers. Chapter 3 showed the groups differ, but not which email works best.

Problem / Purpose
Find out whether changing an email (text color, adding a picture) causes more revenue.
H0: the change has no effect. H1: the change affects revenue. Reject H0 if p < 0.05.

Approach / Solution
A/B testing: split customers into two groups, send each a different email, compare revenue with a t-test, then measure effect size and statistical power.

Tools
pandas, NumPy, SciPy (t-test), statsmodels (power analysis)

Implementation

In [1]:
!wget -q https://bradfordtuckfield.com/desktop.csv
!wget -q https://bradfordtuckfield.com/laptop.csv
!wget -q https://bradfordtuckfield.com/emailresults1.csv
!wget -q https://bradfordtuckfield.com/emailresults2.csv

In [2]:
import pandas as pd
desktop=pd.read_csv('desktop.csv')
laptop=pd.read_csv('laptop.csv')

In [3]:
print(desktop.head())

   userid  spending  age  visits
0       1      1250   31     126
1       2       900   27       5
2       3         0   30     459
3       4      2890   22      18
4       5      1460   38      20


In [4]:
print(laptop.head())

   userid  spending  age  visits
0      31      1499   32      12
1      32       799   23      40
2      33      1200   45      22
3      34         0   59     126
4      35      1350   17      85


In [5]:
import scipy.stats
print(scipy.stats.ttest_ind(desktop['spending'],laptop['spending']))
print(scipy.stats.ttest_ind(desktop['age'],laptop['age']))
print(scipy.stats.ttest_ind(desktop['visits'],laptop['visits']))

TtestResult(statistic=np.float64(-2.109853741030508), pvalue=np.float64(0.03919630411621095), df=np.float64(58.0))
TtestResult(statistic=np.float64(-0.7101437106800108), pvalue=np.float64(0.4804606394128761), df=np.float64(58.0))
TtestResult(statistic=np.float64(0.20626752311535543), pvalue=np.float64(0.8373043059847984), df=np.float64(58.0))


First test: split desktop customers by age (Group A younger gets black text, Group B older gets blue text).

In [6]:
import numpy as np
medianage=np.median(desktop['age'])
groupa=desktop.loc[desktop['age']<=medianage,:]
groupb=desktop.loc[desktop['age']>medianage,:]

In [7]:
emailresults1=pd.read_csv('emailresults1.csv')

In [8]:
print(emailresults1.head())

   userid  revenue
0       1      100
1       2        0
2       3       50
3       4      550
4       5      175


In [9]:
groupa_withrevenue=groupa.merge(emailresults1,on='userid')
groupb_withrevenue=groupb.merge(emailresults1,on='userid')

In [10]:
print(scipy.stats.ttest_ind(groupa_withrevenue['revenue'],groupb_withrevenue['revenue']))

TtestResult(statistic=np.float64(-2.186454851070545), pvalue=np.float64(0.03730073920038287), df=np.float64(28.0))


In [11]:
print(np.mean(groupb_withrevenue['revenue'])-np.mean(groupa_withrevenue['revenue']))

125.0


This result is confounded: Group B is also older, so the $125 could come from age instead of text color. Fix: assign groups randomly.

In [12]:
np.random.seed(18811015)
laptop.loc[:,'groupassignment1']=1*(np.random.random(len(laptop.index))>0.5)
groupc=laptop.loc[laptop['groupassignment1']==0,:].copy()
groupd=laptop.loc[laptop['groupassignment1']==1,:].copy()

In [13]:
emailresults2=pd.read_csv('emailresults2.csv')
groupc_withrevenue=groupc.merge(emailresults2,on='userid')
groupd_withrevenue=groupd.merge(emailresults2,on='userid')

In [14]:
print(scipy.stats.ttest_ind(groupc_withrevenue['revenue'],groupd_withrevenue['revenue']))
print(np.mean(groupd_withrevenue['revenue'])-np.mean(groupc_withrevenue['revenue']))

TtestResult(statistic=np.float64(-2.381320497676198), pvalue=np.float64(0.024288828555138562), df=np.float64(28.0))
260.3333333333333


Effect size: compare $125 to the standard deviation of different data (Cohen's d).

In [15]:
gdps=[365303000000,65994000000,220000000]
print(np.std(gdps))
print(125/np.std(gdps))

burgers=[9.0,12.99,10.50]
print(np.std(burgers))
print(125/np.std(burgers))

print(125/np.std(emailresults1['revenue']))

158884197328.32672
7.867365169217765e-10
1.6455394252341695
75.96293232671214
0.763769235188029


Statistical power: the chance a test detects a real effect.

In [16]:
from statsmodels.stats.power import TTestIndPower
alpha=0.05
nobs=45
effectsize=0.5
analysis = TTestIndPower()
power = analysis.solve_power(effect_size=effectsize, nobs1=nobs, alpha=alpha)
print(power)

0.650185508398425


In [17]:
analysis = TTestIndPower()
alpha = 0.05
effect = 0.5
power = 0.8
observations = analysis.solve_power(effect_size=effect, power=power, alpha=alpha)
print(observations)

63.76561058785403


Conclusion
A/B testing creates new data to test a specific change. Splitting by age was confounded; random assignment fixed it, and adding a picture raised revenue by about $260 per user. Cohen's d (0.76) showed a large effect. Power analysis showed we need about 64 people per group for 80% power. Keep testing (champion/challenger), distrust results that look too good (Twyman's law), and consider ethics.